## Objective

This notebook covers Sprint 2 tasks:
- **Task 5:** Data preprocessing — feature selection and normalization
- **Task 6:** Define the optimal number of clusters (Elbow Method + Silhouette Score)
- **Task 7:** Train the K-Means model and assign clusters
- **Task 8:** Interpret and name the identified profiles

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42
np.random.seed(SEED)

DATA_DIR = "../data"
SYNTHETIC_PATH = os.path.join(DATA_DIR, "synthetic_dataset.csv")
REAL_DATA_PATH = os.path.join(DATA_DIR, "real_data.csv")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10,5)

print("Setup complete.")

Setup complete.


## 2. Load Dataset

In [2]:
USE_REAL_DATA = os.path.exists(REAL_DATA_PATH)

if USE_REAL_DATA:
    df = pd.read_csv(REAL_DATA_PATH)
    print(f"Real dataset loaded: {df.shape[0]} students, {df.shape[1]} columns")
else:
    df = pd.read_csv(SYNTHETIC_PATH)
    print(f"Synthetic dataset loaded: {df.shape[0]} students, {df.shape[1]} columns")

df.head()

Synthetic dataset loaded: 120 students, 10 columns


,student_name,linguistic_score,logical_math_score,spatial_score,bodily_kinesthetic_score,interpersonal_score,intrapersonal_score,emotional_regulation,engagement_frequency,generated_archetype
0,Brenda Alves,7.9,6.7,6.7,5.8,8.7,7.2,7.0,8.3,communicative
1,Sra. Isabelly Câmara,5.9,6.7,6.7,6.4,6.9,7.3,8.5,7.1,balanced
2,Cauã Rocha,7.2,6.9,5.5,7.0,7.0,9.0,6.8,7.2,balanced
3,Dra. Aurora Pastor,5.5,4.6,7.9,9.1,7.6,6.3,6.6,7.4,kinesthetic
4,Ana Beatriz Alves,7.5,10.0,7.7,5.0,5.6,6.6,5.8,7.1,analytical


## 3. Preprocessing

### 3.1 Feature Selection

Two columns are excluded from the model:
- `student_name` - identifier, not a feature
- `generated_archetype` - generation artifact, must not be used as model input

The original scores are kept in `df` for cluster interpretation later.

In [ ]:
score_columns = [
    "linguistic_score",
    "logical_math_score",
    "spatial_score",
    "bodily_kinesthetic_score",
    "interpersonal_score",
    "intrapersonal_score",
    "emotional_regulation",
    "engagement_frequency"
]

X = df[score_columns].copy()

print(f"Features selected: {len(score_columns)}")
print(f"Dataset shape for modeling: {X.shape}")

### 3.2 Missing Value Treatment

Missing values in score columns are filled with the column mean.
This ensures the dataset remains complete without removing student records.

> **Note:** This treatment was already applied during the exploratory analysis
> in `01_exploratory_analysis.ipynb`. It is intentionally repeated here because
> both notebooks are independent — if this notebook is executed directly with
> real data, without running the EDA first, the treatment must still be guaranteed
> before the model receives the data.